**Recitation 0: Checkpointing**

We will show you how to checkpoint and load your model :D

**Section 0: Setup**

Let's define a quick dummy model, optimizer, and scheduler that we'll be saving and loading :)

In [1]:
!pip install wandb --quiet # Install WandB

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
import os

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu" )
print("Device: ", device)

os.environ["WANDB_API_KEY"] = "" # your key here
wandb.login()

/Users/<user>/miniconda3/envs/idl/lib/python3.13/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Device:  mps


wandb: Currently logged in as: <username> (<entity>) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
# Change the following placeholders

RUN_NAME = "my-run-name"                # <-- Change me
PROJECT = "wandb-quickstart"            # <-- Change me
ENTITY = None                           # <-- leave as None for your personal account.
                                        # <-- Set to a team name only for team projects.

run = wandb.init(
        project=PROJECT,
        name=RUN_NAME,
        entity=ENTITY,
        )

wandb: Tracking run with wandb version 0.23.1
wandb: Run data is saved locally in <working-dir>/wandb/run-<timestamp>-<run-id>
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run my-run-name
wandb: ⭐️ View project at https://wandb.ai/<entity>/wandb-quickstart
wandb: 🚀 View run at https://wandb.ai/<entity>/wandb-quickstart/runs/<run-id>


In [4]:
print(f"Run ID: {run.id}")
print(f"Run Name: {run.name}")
print(f"Entity: {run.entity}")
print(f"Run Path: {run.entity}/{run.project}/{run.id}")

Run ID: <run-id>
Run Name: my-run-name
Entity: <entity>
Run Path: <entity>/wandb-quickstart/<run-id>


In [5]:
from collections import OrderedDict

# A simple submodule
class DummySubmodule(nn.Module):
    def __init__(self):
        super(DummySubmodule, self).__init__()
        self.layer = nn.Linear(in_features = 32, out_features = 10)

    def forward(self, x):
        return self.layer(x)

# A simple network
class DummyNetwork(nn.Module):

    def __init__(self):
        super(DummyNetwork, self).__init__()

        self.model = nn.Sequential(
                OrderedDict([
                    ("expand",   nn.Sequential(nn.Linear(32,64), nn.ReLU(), nn.Linear(64,128), nn.ReLU())),
                    ("contract", nn.Sequential(nn.Linear(128,64), nn.ReLU(), nn.Linear(64,32), nn.ReLU())),
                    ])
                )

        self.dummy = DummySubmodule()

    def forward(self, x):
        res = self.model(x)
        res = self.dummy(res)
        return res

# Declare the model, optimizer, and scheduler
model = DummyNetwork().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

Let's take a look at some of the information we can checkpoint

In [6]:
# Print model's state_dict
print("==============================================================")
print("Model's state_dict:")
for param_tensor in model.state_dict():
    print(param_tensor, "\t", model.state_dict()[param_tensor].size())
print("==============================================================")
# Print optimizer's state_dict
print("Optimizer's state_dict:")
for var_name in optimizer.state_dict():
    print(var_name, "\t", optimizer.state_dict()[var_name])
print("==============================================================")
# Print scheduler's state_dict
print("\nScheduler's state_dict:")
for var_name, value in scheduler.state_dict().items():
    print(var_name, "\t", value)
print("==============================================================")

Model's state_dict:
model.expand.0.weight 	 torch.Size([64, 32])
model.expand.0.bias 	 torch.Size([64])
model.expand.2.weight 	 torch.Size([128, 64])
model.expand.2.bias 	 torch.Size([128])
model.contract.0.weight 	 torch.Size([64, 128])
model.contract.0.bias 	 torch.Size([64])
model.contract.2.weight 	 torch.Size([32, 64])
model.contract.2.bias 	 torch.Size([32])
dummy.layer.weight 	 torch.Size([10, 32])
dummy.layer.bias 	 torch.Size([10])
Optimizer's state_dict:
state 	 {}
param_groups 	 [{'lr': 0.01, 'momentum': 0, 'dampening': 0, 'weight_decay': 0, 'nesterov': False, 'maximize': False, 'foreach': None, 'differentiable': False, 'fused': None, 'initial_lr': 0.01, 'params': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]}]

Scheduler's state_dict:
step_size 	 10
gamma 	 0.1
base_lrs 	 [0.01]
last_epoch 	 0
_step_count 	 1
_is_initial 	 False
_get_lr_called_within_step 	 False
_last_lr 	 [0.01]


#######################################################################################################

**Section 1: How to save the checkpoint Saving a checkpoint**

Checkpointing locally

In [7]:
# let's pretend we're in the middle of our training
epoch = 6 # pretend we're in our 6th epoch
loss = 0.78 #pretend this is our model's loss at the moment

CHECKPOINT_DIR = "."
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoint_path = os.path.join(CHECKPOINT_DIR ,f"{RUN_NAME}_{epoch}.pth")

# Saving your states locally with torch.save
torch.save({
    'model_state_dict': model.state_dict(),   # saving the model state
    # if isinstance(model, nn.DataParallel) 'model_state_dict': model.module.state_dict()
    'optimizer_state_dict': optimizer.state_dict(),   # saving the optimizer state
    'scheduler_state_dict': scheduler.state_dict(),   # saving the scheduler state
    'epoch': epoch,
    'current_loss': loss,
    "wandb_run_id": run.id,
    }, checkpoint_path
)

Alternative way of Saving Model Checkpoints Using Helper Function(an example from HW)

In [8]:
# Alternative approach: Helper function method (commonly used in HW assignments)
# This provides a cleaner, reusable way to save all checkpoint components
def save_model(model, optimizer, scheduler, metrics, epoch, path, run_id=None):
    """
    Helper function to save model checkpoint locally.
    
    Args:
        model: The neural network model
        optimizer: The optimizer
        scheduler: The learning rate scheduler
        metrics: Training metrics (e.g., loss, accuracy)
        epoch: Current epoch number
        path: File path to save the checkpoint
    """
    torch.save(
        {"model_state_dict"         : model.state_dict(),
         "optimizer_state_dict"     : optimizer.state_dict(),
         "scheduler_state_dict"     : scheduler.state_dict(),
         "metric"                   : metrics,
         "epoch"                    : epoch,
         "wandb_run_id"             : run_id,
         },
         path)
    print(f"Checkpoint saved to {path}")

# Example usage:
metrics = {'loss': 0.78, 'accuracy': 0.92}
save_model(model, optimizer, scheduler, metrics, epoch=6, path="checkpoint_epoch6.pth")

Checkpoint saved to checkpoint_epoch6.pth


Checkpointing and saving to wandb as an artifact

In [9]:
# The run was already started back in Section 0. We're still logging into it.
# In a real training script, wandb.init() goes at the top and everything below goes into your training loop.

# run = wandb.init(
#     project="wandb-quickstart",
#     name="<run_name>",
#     )

# ...
# ...
# ...
# Within a training loop (or wherever else you want)....

# Option 1:
# create artifacts (keeps track of versioning, and is much more organized to work with between collaborators)
checkpoint_artifact = wandb.Artifact(RUN_NAME, type="checkpoint") # You can switch type="model if you only want to save a model"

checkpoint_artifact.add_file(checkpoint_path)

run.log_artifact(checkpoint_artifact)

# Option 2:
# directly save the model to wandb
wandb.save(checkpoint_path, base_path=CHECKPOINT_DIR)

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


['<working-dir>/wandb/run-<timestamp>-<run-id>/files/my-run-name_6.pth']

#######################################################################################################

**Section 2\: Loading a checkpoint file into our current model**

Downloading a model from wandb

In [ ]:
# METHOD 1: Download from wandb Artifact
# If you need to re-obtain the run, you can do the following....
api = wandb.Api()

# information can be obtained from the wandb link address as follows:
# https://wandb.ai/<USERNAME>/<PROJECT_NAME>/runs/<RUN_ID>?nw=nwuser<USERNAME>
# <USERNAME> is your ENTITY

# Unlike ENTITY and PROJECT, this one cannot be auto-filled
# If you are here, your local files are gone, so `run` above is a NEW run with a NEW id. Paste from the URL:
RUN_ID = "<paste-from-url>"
api_run = api.run(f"{ENTITY}/{PROJECT}/{RUN_ID}")

# To retrieve the artifact....
# Get the artifact (choose which version of the checkpoint you want)
artifact = api_run.use_artifact(f"{RUN_NAME}:latest")
# Downloading the artifact
artifact_dir = artifact.download()
# Loading the checkpoint dict
checkpoint_filename = os.path.basename(checkpoint_path)
checkpoint_dict = torch.load(os.path.join(artifact_dir, checkpoint_filename))

Loading a .pth checkpoint file from our local directory to our model

In [10]:
# .pth checkpoint file path can also be obtained from a locally saved .pth file. Or, you can use
# the checkpoint_dict obtained from the prior wandb artifact download :)
checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}_6.pth")
checkpoint_dict = torch.load(checkpoint_path)

# Loading model weights
# if isinstance(model, nn.DataParallel) model.module.load_state_dict(checkpoint_dict['model_state_dict'])
model.load_state_dict(checkpoint_dict['model_state_dict'])
# Loading optimizer state
optimizer.load_state_dict(checkpoint_dict['optimizer_state_dict'])
# Loading the scheduler state
scheduler.load_state_dict(checkpoint_dict['scheduler_state_dict'])
# find the epoch we left off at
current_epoch = checkpoint_dict['epoch']
# Find any metrics that might be relevant
current_loss = checkpoint_dict['current_loss']

# ---- Resume the SAME wandb run if the checkpoint knows one, else start fresh ----
resume_id = checkpoint_dict.get("wandb_run_id")         # .get() returns None when the key wasn't saved in the ckct

# Close runs this notebook already opened
wandb.finish()

run = wandb.init(
        project=PROJECT,
        name=RUN_NAME,
        entity=ENTITY,
        id=resume_id,
        resume="must" if resume_id else None,
        )

if resume_id:
    print(f"Resumed run {resume_id} from epoch {current_epoch}")
else:
    print(f"No run id in checkpoint -- started new run {run.id}")

wandb: updating run metadata
wandb: uploading config.yaml
wandb: uploading summary
wandb: 🚀 View run my-run-name at: https://wandb.ai/<entity>/wandb-quickstart/runs/<run-id>
wandb: ⭐️ View project at: https://wandb.ai/<entity>/wandb-quickstart
wandb: Synced 4 W&B file(s), 0 media file(s), 2 artifact file(s) and 1 other file(s)
wandb: Find logs at: ./wandb/run-<timestamp>-<run-id>/logs
wandb: setting up run <run-id>
wandb: Tracking run with wandb version 0.23.1
wandb: Run data is saved locally in <working-dir>/wandb/run-<timestamp>-<run-id>
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run my-run-name
wandb: ⭐️ View project at https://wandb.ai/<entity>/wandb-quickstart
wandb: 🚀 View run at https://wandb.ai/<entity>/wandb-quickstart/runs/<run-id>


Resumed run <run-id> from epoch 6


Alternative way of Loading Model Checkpoints Using Helper Function(an example from HW)

In [11]:

# Alternative approach: Helper function method (commonly used in HW assignments)
# This provides a cleaner, reusable way to load checkpoints with optional components
def load_model(model, optimizer=None, scheduler=None, path='./checkpoint.pth'):
    """
    Helper function to load model checkpoint from local storage.
    
    Args:
        model: The neural network model to load weights into
        optimizer: The optimizer (optional, can be None for inference)
        scheduler: The learning rate scheduler (optional, can be None)
        path: File path of the checkpoint to load
    
    Returns:
        model, optimizer, scheduler, epoch, metrics
    """
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    else:
        optimizer = None
    
    if scheduler is not None:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    else:
        scheduler = None
    
    epoch = checkpoint['epoch']
    metrics = checkpoint['metric']
    
    print(f"Checkpoint loaded from {path}")
    print(f"Resuming from epoch {epoch} with metrics: {metrics}")
    
    return model, optimizer, scheduler, epoch, metrics

# Example usage 1: Load for resuming training (with optimizer and scheduler)
model, optimizer, scheduler, epoch, metrics = load_model(
    model, 
    optimizer=optimizer, 
    scheduler=scheduler, 
    path="checkpoint_epoch6.pth"
)

# Example usage 2: Load for inference only (no optimizer or scheduler needed)
model, _, _, epoch, metrics = load_model(
    model, 
    optimizer=None, 
    scheduler=None, 
    path="checkpoint_epoch6.pth"
)

Checkpoint loaded from checkpoint_epoch6.pth
Resuming from epoch 6 with metrics: {'loss': 0.78, 'accuracy': 0.92}
Checkpoint loaded from checkpoint_epoch6.pth
Resuming from epoch 6 with metrics: {'loss': 0.78, 'accuracy': 0.92}


In [12]:
# If you want to load specific parts of your model (in our case, we can load just the lower layers or just the upper layers)
specific_weights = { # Creates dictionary of only desired weights
    key: value
    for key, value in checkpoint_dict['model_state_dict'].items()
    if 'expand' in key
}

model.load_state_dict(specific_weights, strict=False)

_IncompatibleKeys(missing_keys=['model.contract.0.weight', 'model.contract.0.bias', 'model.contract.2.weight', 'model.contract.2.bias', 'dummy.layer.weight', 'dummy.layer.bias'], unexpected_keys=[])